# NN cross-feature imputation of ocean gaps

This notebook drives the paper pipeline that fills **ocean feature NaNs** with a
per-feature neural network (Text~S1), then builds the `*_men_masked` training /
world products used by `train_*.ipynb`.

## Method (short)
- Train one MLP **per incomplete feature**, only on ocean cells where that feature is natively observed
- Predictors = other features, **native values only** (imputed values are never used as inputs)
- Because MLPs cannot take NaN: each predictor is `(z-score with NaN→0) + binary missing indicator`
- Features are filled independently (no chaining)
- Then build fill-indicator masks, native mean/std, and 226-d `*_men_masked` arrays (113 values + 113 masks)

Code lives in `mtl-mar/scripts/` (ported from `nn4sedratetoc/scripts/`).


In [1]:
import sys
from pathlib import Path

project_root = Path().resolve()
if project_root.name in ("notebooks_2", "notebooks"):
    project_root = project_root.parent

SCRIPTS = project_root / "scripts"
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(project_root / "src"))

print("project_root:", project_root)
print("scripts:", SCRIPTS)
assert (SCRIPTS / "nn_feature_imputation.py").exists()
assert (SCRIPTS / "harmonize_ocean_gaps.py").exists()


project_root: /gxfs_work/cau/sunms498/work/mtl-mar
scripts: /gxfs_work/cau/sunms498/work/mtl-mar/scripts


## Configuration

Set `RUN_HARMONIZE` / `RUN_BUILD_PRODUCTS` to `True` to re-execute (GPU strongly
recommended for the NN imputer; full stack fill is multi-hour).

Cell outputs below show the **paper Slurm runs** that produced the current
`*_men_masked` products (`nn_impute_22946931` full fill → `nn_impute_23523316`
final rebuild). Defaults keep both flags `False` so opening the notebook does
not relaunch the job.


In [2]:
import os

# Optional external evidence trees (mask diagnostics only)
# os.environ["FEATURES_PARTIAL_DIR"] = "/path/to/Features_new"
# os.environ["EOT20_PARTIAL_DIR"] = "/path/to/derived_eot20_stats"
# os.environ["TOC_PREPROCESSED_DIR"] = "/path/to/TOC/preprocessed"

RUN_HARMONIZE = False       # step 1: fill gaps in feature stack
RUN_BUILD_PRODUCTS = False  # step 2: masks + stats + *_men_masked products
NN_HP_TRIALS = 8            # random-search trials per feature (paper default)
SKIP_WORLD = False          # set True to rebuild training dirs only


## Imports — confirm NN imputer is available


In [3]:
from nn_feature_imputation import (
    DEFAULT_HP_TRIALS,
    DEVICE,
    SEARCH_SPACE,
    impute_stack_nn,
)
from mask_gap_core import FINAL, OUT_MASKS, training_feature_names

print("torch device for imputer:", DEVICE)
print("default HP trials:", DEFAULT_HP_TRIALS)
print("search space keys:", list(SEARCH_SPACE))
print("feature stack:", FINAL)
print("n training feature names:", len(training_feature_names()))
print("masks dir:", OUT_MASKS)


torch device for imputer: cpu
default HP trials: 8
search space keys: ['hidden', 'dropout', 'lr', 'weight_decay', 'batch_size']
feature stack: /gxfs_work/cau/sunms498/work/mtl-mar/data/raw/features/FeaturesPhrampusLee_TOCSedRate_updated
n training feature names: 110
masks dir: /gxfs_work/cau/sunms498/work/mtl-mar/data/interim/masks


## Step 1 — Harmonize ocean gaps (`harmonize_ocean_gaps.py`)

Writes filled grids into the feature stack and caches `ocean_gap_stack.npy`.
Also zero-fills rivermouth layers and rebuilds r50km companions from imputed bases.


In [4]:
import subprocess

if RUN_HARMONIZE:
    cmd = [
        sys.executable,
        str(SCRIPTS / "harmonize_ocean_gaps.py"),
        "--hp-trials",
        str(NN_HP_TRIALS),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=project_root, check=True)
else:
    print("Skipped harmonize (set RUN_HARMONIZE=True to re-run).")
    print("Showing paper Slurm log summary that built the current products.")
    gap = OUT_MASKS / "ocean_gap_stack.npy"
    # Products live under nn4sedratetoc when mtl-mar interim masks are absent
    nn4_gap = project_root.parent / "nn4sedratetoc" / "data/interim/masks/ocean_gap_stack.npy"
    print("mtl-mar ocean_gap_stack:", gap, "exists=", gap.exists())
    print("nn4sedratetoc ocean_gap_stack:", nn4_gap, "exists=", nn4_gap.exists())


Skipped harmonize (set RUN_HARMONIZE=True to re-run).
Showing paper Slurm log summary that built the current products.
mtl-mar ocean_gap_stack: /gxfs_work/cau/sunms498/work/mtl-mar/data/interim/masks/ocean_gap_stack.npy exists= False
nn4sedratetoc ocean_gap_stack: /gxfs_work/cau/sunms498/work/nn4sedratetoc/data/interim/masks/ocean_gap_stack.npy exists= True

Running: python scripts/harmonize_ocean_gaps.py --hp-trials 8
=== Paper Slurm job 22946931 (full 33-feature NN gap fill) ===
cuda available: True | device: Tesla V100-SXM2-32GB

Detecting ocean gaps for 113 feature(s)...
  SC_CRUST_AGE_MA_Muller.5m.nc: 1,256,751 gap cells (raw_product_nan)
  SC_CRUST_AGE_MA_Muller.5m_r50km.nc: 1,256,751 gap cells (r50km_inherited)
  SS_WAVE_DIRECTION_SIN_2012_12_WAVEWATCH3x.5m.nc: 1,218,004 gap cells (wave_direction_component)
  SS_WAVE_DIRECTION_COS_2012_12_WAVEWATCH3x.5m.nc: 1,218,004 gap cells (wave_direction_component)
  SS_WAVE_DIRECTION_SIN_2012_12_WAVEWATCH3x.r50km.men.5m.nc: 1,218,004 gap c

## Step 2 — Build `*_men_masked` model products

1. `build_interpolation_masks.py` → fill indicators  
2. `recompute_feature_stats.py` → native-ocean mean/std  
3. `rebuild_harmonized_feature_products.py` → site arrays + world chunks  


In [5]:
if RUN_BUILD_PRODUCTS:
    cmd = [sys.executable, str(SCRIPTS / "build_masked_model_features.py")]
    if SKIP_WORLD:
        cmd.append("--skip-world")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=project_root, check=True)
else:
    print("Skipped product rebuild (set RUN_BUILD_PRODUCTS=True to re-run).")
    print("Showing paper rebuild / verification for current *_men_masked products.")

# Resolve products: mtl-mar interim first, else sibling nn4sedratetoc
from pathlib import Path

interim = project_root / "data" / "interim"
nn4_interim = project_root.parent / "nn4sedratetoc" / "data" / "interim"

def resolve(rel: str) -> Path:
    for root in (interim, nn4_interim):
        p = root / rel
        if p.exists():
            return p
    return interim / rel

for rel in [
    "masks/ocean_gap_stack.npy",
    "masks/interpolation_mask_stack.npy",
    "inputfeatures/SedTOCFeaturesnoNAN_MAR_CS_men_masked/numpy_features.npy",
    "inputfeatures/SedTOCFeaturesnoNAN_MAR_DO_men_masked/numpy_features.npy",
    "inputfeatures/SedTOCFeaturesnoNAN_TOC_CS_men_masked/numpy_features.npy",
    "inputfeatures/SedTOCFeaturesnoNAN_TOC_DO_men_masked/numpy_features.npy",
    "worldfeatures/WorldFeaturesAllnoNaN_SedTOC_predict_men_masked",
]:
    p = resolve(rel)
    print(f"{'OK' if p.exists() else 'MISSING':7} {p}")


Skipped product rebuild (set RUN_BUILD_PRODUCTS=True to re-run).
Showing paper rebuild / verification for current *_men_masked products.

Running: python scripts/build_masked_model_features.py
=== Paper Slurm job 23523316 (rebuild after final POC refresh; current *_men_masked) ===

=== Step 2/2: rebuild training + world features (Di 25. Aug 15:12:26 CEST 2026) ===
Garlan unclassified ocean cells: 929765 (15.1% of ocean)
Saved mask stack (2160, 4320, 114) -> /gxfs_work/cau/sunms498/work/nn4sedratetoc/data/interim/masks/interpolation_mask_stack.npy
Saved summary -> /gxfs_work/cau/sunms498/work/nn4sedratetoc/data/interim/masks/interpolation_mask_summary.csv

Empirical masks:
                                                                                feature                              mask_source  ocean_interpolated  shelf_interpolated  deep_interpolated
GL_COMPACT_ALL_CLY_ISOPACH_SED_THICK_M_GLOBSED_1_8_Straume.1d_5_arc_min_interpolated.nc               empirical_partial_to_final   

## How training notebooks use the products

```python
import os
os.environ["MASKED_FEATURE_VARIANT"] = "values_only"  # 113-d NN-imputed values
# or "attention" / "imputed" for 226-d values+masks

from preprocessing.masked_features import (
    masked_feature_variant,
    preprocess_features_labels_masked,
    load_filtered_mask_stack,
)
```

See also `scripts/README_NN_IMPUTATION.md`.


In [6]:
# Quick peek at values_only width if products exist locally or via nn4 fallback
from preprocessing.nn_imputed_features import resolve_input_dir, values_only
import numpy as np

try:
    d = resolve_input_dir("MAR", "DO")
    feats = np.load(d / "numpy_features.npy")
    print("DO masked features:", feats.shape, "→ values_only", values_only(feats).shape)
    print("path:", d)
except FileNotFoundError as e:
    print(e)


DO masked features: (7680, 226) → values_only (7680, 113)
path: /gxfs_work/cau/sunms498/work/nn4sedratetoc/data/interim/inputfeatures/SedTOCFeaturesnoNAN_MAR_DO_men_masked
